In [21]:
from qdrant_client import QdrantClient   
from qdrant_client.models import Distance, VectorParams, PointStruct

import pandas as pd
import openai

## Read the sampled dataset with Amazon inventory data

In [22]:
df_items = pd.read_json(
    "/Users/benny/AI-Engineer/data/meta_Books_fiction_rating_filtered_subset.jsonl", lines=True
)

In [23]:
df_items.head()

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together
0,Books,Conqueror's Kiss,"Mass Market Paperback – November 1, 2006",{'avatar': 'https://m.media-amazon.com/images/...,4.4,450,[There is little hope for sanctuary in the mid...,[],1.35,[{'large': 'https://m.media-amazon.com/images/...,[],Hannah Howell (Author),"[Books, Literature & Fiction, Genre Fiction]","{'Publisher': 'Zebra (November 1, 2006)', 'Lan...",0821780050,NaN
1,Books,Rebel Hunter (CERBERUS),"Paperback – October 26, 2020",{'avatar': 'https://m.media-amazon.com/images/...,4.7,351,[Nolan Garrett has just declared a war on terr...,[],4.27,[{'large': 'https://m.media-amazon.com/images/...,[],Andy Peloquin (Author),"[Books, Science Fiction & Fantasy, Science Fic...",{'Publisher': 'Independently published (Octobe...,B08M2BKBNG,NaN
2,Buy a Kindle,Truth is the Soul of the Sun - A Biographical ...,NaN,{'avatar': 'https://m.media-amazon.com/images/...,3.9,32,[REVISED SECOND EDITIONThis is the story of Ha...,"[Review, Truth is the Soul of the Sun, Maria I...",0.0,[{'large': 'https://m.media-amazon.com/images/...,[],Maria Isabel Pita (Author) Format: Kindle Ed...,"[Books, Literature & Fiction, Genre Fiction]",{'Publisher': 'Lucid Press; 2nd edition (July ...,B002HWRUFW,NaN
3,Books,Iphigeneia at Aulis (Greek Tragedy in New Tran...,Reprint Edition,{'avatar': 'https://m.media-amazon.com/images/...,3.9,32,[Merwin and Dimock have provided a new transla...,"[Review, ""Wonderful edition.""―Professor Coni M...",9.18,[{'large': 'https://m.media-amazon.com/images/...,[],"Euripides (Author), W. S. Merwin (Translator)...","[Books, Literature & Fiction, History & Critic...",{'Publisher': 'Oxford University Press; Reprin...,0195077091,NaN
4,Books,The Rowan,"Hardcover – August 20, 1990",{'avatar': 'https://m.media-amazon.com/images/...,4.7,1368,"[The Rowan, a beautiful and powerful psychic w...","[From Publishers Weekly, McCaffrey leaves behi...",24.01,[{'large': 'https://m.media-amazon.com/images/...,[],Anne McCaffrey (Author),"[Books, Science Fiction & Fantasy, Science Fic...",{'Publisher': 'Putnam Adult; First Edition (Au...,0399135707,NaN


In [24]:
empty_descriptions = df_items["description"].str.len().eq(0)

print(empty_descriptions.sum())

504


In [25]:
list(df_items['features'].items())[0]

(0,
 ['There is little hope for sanctuary in the midst of the tumultuous Scottish border wars, yet one woman may find safe refuge...\xa0 in the arms of her sworn enemy...',
  'Jennet Graeme has witnessed terrible tragedy during the many years of strife between the Scots and the English. As Scottish invaders plunder her convent sanctuary, she defiantly resists the blond warrior who claims her as his prize. But his brute strength is overpowering and Jennet is forced to ride with him through the lawless lands, tending to the wounded, protected and desired by a man she wants to hate... but cannot...',
  "Sir Hacon Gillard is moved by Jennet's compassion and mercy. As a loyal knight, he's pledged fealty to his king's command, even as he loses his heart to this remarkable woman. Merciless in combat... yet there burns within him a spark for something far beyond the heat of battle..."])

In [26]:
empty_images = df_items["images"].str.len().eq(0)

print(empty_images.sum())

284


In [27]:
list(df_items['images'].items())[0:10]

[(0,
  [{'large': 'https://m.media-amazon.com/images/I/51QK4k8PimL._SY291_BO1,204,203,200_QL40_FMwebp_.jpg',
    'variant': 'MAIN'}]),
 (1,
  [{'large': 'https://m.media-amazon.com/images/I/51Pbtp7o4tL._SX326_BO1,204,203,200_.jpg',
    'variant': 'MAIN'}]),
 (2,
  [{'large': 'https://m.media-amazon.com/images/I/41HMu6eIDlL.jpg',
    'variant': 'MAIN'}]),
 (3,
  [{'large': 'https://m.media-amazon.com/images/I/41o2HIFBESL._SY291_BO1,204,203,200_QL40_FMwebp_.jpg',
    'variant': 'MAIN'}]),
 (4,
  [{'large': 'https://m.media-amazon.com/images/I/511yvDwZBPL._SX345_BO1,204,203,200_.jpg',
    'variant': 'MAIN'}]),
 (5,
  [{'large': 'https://m.media-amazon.com/images/I/518ZBPngBVL._SX326_BO1,204,203,200_.jpg',
    'variant': 'MAIN'}]),
 (6,
  [{'large': 'https://m.media-amazon.com/images/I/51RxWwTcdLL._SY291_BO1,204,203,200_QL40_FMwebp_.jpg',
    'variant': 'MAIN'}]),
 (7, []),
 (8, []),
 (9, [])]

#### Looks like there are a lot of empty description and pictures, we don't want those for the embedded data set

In [28]:
df_filtered = df_items[~empty_descriptions & ~empty_images]

In [29]:
len(df_filtered)

411

## Preprocess title and features

In [30]:
def preprocess_description(row):
    return f"{row['title']}{' '.join(row['features'])}"

In [31]:
def get_image_url(row):
    return row['images'][0].get('large')

In [32]:
df_filtered['description'] = df_filtered.apply(preprocess_description, axis=1)

In [33]:
df_filtered['image_urk'] = df_filtered.apply(get_image_url, axis=1)

In [34]:
df_filtered.head()

,main_category,title,subtitle,author,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,image_urk
2,Buy a Kindle,Truth is the Soul of the Sun - A Biographical ...,NaN,{'avatar': 'https://m.media-amazon.com/images/...,3.9,32,[REVISED SECOND EDITIONThis is the story of Ha...,Truth is the Soul of the Sun - A Biographical ...,0.0,[{'large': 'https://m.media-amazon.com/images/...,[],Maria Isabel Pita (Author) Format: Kindle Ed...,"[Books, Literature & Fiction, Genre Fiction]",{'Publisher': 'Lucid Press; 2nd edition (July ...,B002HWRUFW,NaN,https://m.media-amazon.com/images/I/41HMu6eIDl...
3,Books,Iphigeneia at Aulis (Greek Tragedy in New Tran...,Reprint Edition,{'avatar': 'https://m.media-amazon.com/images/...,3.9,32,[Merwin and Dimock have provided a new transla...,Iphigeneia at Aulis (Greek Tragedy in New Tran...,9.18,[{'large': 'https://m.media-amazon.com/images/...,[],"Euripides (Author), W. S. Merwin (Translator)...","[Books, Literature & Fiction, History & Critic...",{'Publisher': 'Oxford University Press; Reprin...,0195077091,NaN,https://m.media-amazon.com/images/I/41o2HIFBES...
4,Books,The Rowan,"Hardcover – August 20, 1990",{'avatar': 'https://m.media-amazon.com/images/...,4.7,1368,"[The Rowan, a beautiful and powerful psychic w...","The RowanThe Rowan, a beautiful and powerful p...",24.01,[{'large': 'https://m.media-amazon.com/images/...,[],Anne McCaffrey (Author),"[Books, Science Fiction & Fantasy, Science Fic...",{'Publisher': 'Putnam Adult; First Edition (Au...,0399135707,NaN,https://m.media-amazon.com/images/I/511yvDwZBP...
6,Books,Taras Bulba (Modern Library Classics),NaN,{'avatar': 'https://m.media-amazon.com/images/...,4.1,136,"[The First New Translation in Forty Years, Set...",Taras Bulba (Modern Library Classics)The First...,None,[{'large': 'https://m.media-amazon.com/images/...,[],"Nikolai Gogol (Author), Peter Constantine (Tr...","[Books, Literature & Fiction, Genre Fiction]","{'Publisher': 'Modern Library (December 30, 20...",0812971191,NaN,https://m.media-amazon.com/images/I/51RxWwTcdL...
11,Books,Dr. Hugh Mann (Zombie Fallout),"Paperback – June 28, 2011",{'avatar': 'https://m.media-amazon.com/images/...,4.5,699,[Before there were zombies.....There was the v...,Dr. Hugh Mann (Zombie Fallout)Before there wer...,8.24,[{'large': 'https://m.media-amazon.com/images/...,[],Mark Tufo (Author),"[Books, Literature & Fiction, Genre Fiction]",{'Publisher': 'CreateSpace Independent Publish...,1463669364,NaN,https://m.media-amazon.com/images/I/51Q4591AJv...


#### Define the embedding function

In [37]:
response = openai.embeddings.create(
    input = 'random text',
    model='text-embedding-3-small'
)

AuthenticationError: Error code: 401 - {'error': {'message': 'Your API key has expired. Create a new API key to continue.', 'type': 'invalid_request_error', 'code': 'expired_secret_key', 'param': None}, 'status': 401}